# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities (record sets, fields, columns, etc.) are referenced explicitly by their `@id` fields, as prescribed by the Croissant schema.

### Dataset Source
This dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset (metadata and structure)
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors: {[a for a in getattr(metadata, 'author', [])]}\n")
print(f"License: {metadata.license}")

## 2. Data Overview

Explore available record sets, fields, and their `@id`s. Each entity with data (record set, field, etc.) is referenced below by its `@id`.


Let's discover what record sets are available in the dataset.

In [ ]:
# List available record sets with their @id and names
record_set_list = dataset.record_sets
print("Available Record Sets:")
for rs in record_set_list:
    print(f"- @id: {rs.id} | name: {rs.name}")

# Preview fields for each record set
for rs in record_set_list:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    Field: {field.name} (@id: {field.id}), dataType: {field.data_type}")
    else:
        print("    No fields found.")

## 3. Data Extraction

Let's load records from a record set, referencing entities by their `@id` fields. We first select the main record set for regression results.

**Note:** If record set `@id`s are not obvious, we enumerate those printed in the previous section.

In [ ]:
# Select the main record set(s) by @id
# Edit record_set_ids to match actual @id values discovered above.
record_set_ids = [rs.id for rs in dataset.record_sets]  # Use all found record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded {len(df)} records for RecordSet: {record_set_id}")
    print(f"    Columns: {df.columns.tolist()}")
    if len(df) > 0:
        print(df.head(3))

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field, filter on a value, normalize it, and optionally group records by another field.

- **All field and record set references use `@id`** as required.
- Edit `numeric_field_id` and `group_field_id` below to match the corresponding field @ids printed in the overview.

In [ ]:
# Example: pick the first available record set and numeric field found
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Display the columns again
print(f"Data columns in {main_record_set_id}: {df.columns.tolist()}")

# Guess numeric field: try first float/int column, else edit the id manually
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.columns[0]  # fallback

print(f"Using numeric field for EDA: {numeric_field_id}")

# EDA: filter, normalize, group
threshold = df[numeric_field_id].mean() if numeric_field_id in df else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (using mean as threshold):")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by the first non-numeric field
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization

Visualize numeric data distributions and relationships. Please adjust the field `@id` below as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If group_field_id exists, boxplot comparing distributions
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We explored the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields.
- The dataset features regression results for knowledge adoption predictors in rangeland management — exploring numeric outputs, filtering, normalization, and group analysis.
- This notebook can be extended with domain-specific analysis or machine learning workflows using the curated tabular data.